# 20260508 Post Curation

This notebook compares pre-BERT term-suggestion lists with post-curation final relationship files and exports, for each disease:

- all names from the pre-BERT list
- one acceptance flag per name: `accepted_in_second_search` (`yes`/`no`)

Outputs are saved as one CSV per disease under the configured output directory.

In [ ]:
from pathlib import Path
import os
import pandas as pd

_LIT_DIR = Path.cwd()
_RELEASE_ROOT = Path(os.environ.get("RELEASE_ROOT", str(_LIT_DIR.parents[1])))
CURATION_ROOT = Path(os.environ.get(
    "CURATION_ROOT",
    str(_RELEASE_ROOT / "dataset" / "PrimeKG-Plus-RD" / "curation_source"),
))
BASE_DIR = CURATION_ROOT
POST_DIR = CURATION_ROOT / "Post curation"
BEFORE_BERT_DIR = POST_DIR / "before_bert"
FINALS_V1_DIR = POST_DIR / "finals_v1"
QC_OUTPUTS_DIR = POST_DIR / "qc_outputs"
INTERMEDIATE_DIR = POST_DIR / "intermediate"

QC_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

DISEASE_FILE_PAIRS = {
    "Canavan": {
        "before": BEFORE_BERT_DIR / "20260502-Canavan_disease_For_term_suggestion_before_BERT.csv",
        "final": FINALS_V1_DIR / "20260508-Canavan_final.csv",
    },
    "Batten": {
        "before": BEFORE_BERT_DIR / "2026058-Batten_disease_For_term_suggestion_before_BERT.csv",
        "final": FINALS_V1_DIR / "20260508-Batten_final.csv",
    },
    "NPC": {
        "before": BEFORE_BERT_DIR / "2026058-NPC_disease_For_term_suggestion_before_BERT.csv",
        "final": FINALS_V1_DIR / "20260508-NMP_final.csv",
    },
}


def _norm_name(x) -> str:
    return str(x).strip().lower()


def _read_before_names(before_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(before_csv)
    if "entity_name" not in df.columns:
        raise ValueError(f"Missing 'entity_name' column in {before_csv}")

    if "original_entity_types" in df.columns:
        out = df[["entity_name", "original_entity_types"]].copy()
        out = out.rename(columns={"original_entity_types": "entity_type"})
    else:
        out = df[["entity_name"]].copy()
        out["entity_type"] = pd.NA

    out = out.dropna(subset=["entity_name"])
    out["entity_name"] = out["entity_name"].astype(str).str.strip()
    out = out[out["entity_name"] != ""]

    out["entity_type"] = out["entity_type"].astype("string").str.strip()
    out.loc[out["entity_type"].isin(["", "nan", "NaN"]), "entity_type"] = pd.NA

    # Keep one row per name; if repeated, merge unique entity types.
    out = (
        out.groupby("entity_name", as_index=False)
        .agg(entity_type=("entity_type", lambda s: "|".join(sorted(set([str(x) for x in s.dropna()]))) if s.notna().any() else pd.NA))
    )

    out["entity_name_norm"] = out["entity_name"].map(_norm_name)
    return out


def _read_final_suggested(final_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(final_csv)

    required_cols = {
        "entity1", "entity2", "entity1_status", "entity2_status",
        "entity1_suggested_name", "entity2_suggested_name",
    }
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns {missing} in {final_csv}")

    side1 = df[["entity1", "entity1_status", "entity1_suggested_name"]].rename(
        columns={
            "entity1": "entity_name",
            "entity1_status": "entity_status",
            "entity1_suggested_name": "suggested_name",
        }
    )
    side2 = df[["entity2", "entity2_status", "entity2_suggested_name"]].rename(
        columns={
            "entity2": "entity_name",
            "entity2_status": "entity_status",
            "entity2_suggested_name": "suggested_name",
        }
    )

    long_df = pd.concat([side1, side2], ignore_index=True)
    long_df = long_df.dropna(subset=["entity_name"])
    long_df["entity_name"] = long_df["entity_name"].astype(str).str.strip()
    long_df = long_df[long_df["entity_name"] != ""]
    long_df["entity_name_norm"] = long_df["entity_name"].map(_norm_name)

    long_df["suggested_name"] = long_df["suggested_name"].astype("string")
    long_df["suggested_name"] = long_df["suggested_name"].str.strip()
    long_df.loc[long_df["suggested_name"].isin(["", "nan", "NaN"]), "suggested_name"] = pd.NA

    return long_df


def build_outputs_for_disease(disease_name: str, before_csv: Path, final_csv: Path, out_dir: Path) -> dict:
    before_df = _read_before_names(before_csv)
    final_long = _read_final_suggested(final_csv)

    # Keep only names that are in the pre-BERT suggestion pool.
    final_in_scope = final_long.merge(
        before_df[["entity_name", "entity_name_norm"]],
        on="entity_name_norm",
        how="inner",
        suffixes=("_final", "_before"),
    )

    # Accepted in second search: has a non-empty suggested name in final output.
    accepted_df = (
        final_in_scope[~final_in_scope["suggested_name"].isna()]
        .groupby("entity_name_before", as_index=False)
        .agg(
            suggested_name=("suggested_name", lambda s: "|".join(sorted(set([str(x) for x in s.dropna()])))),
            observed_statuses=("entity_status", lambda s: "|".join(sorted(set([str(x) for x in s.dropna()])))),
        )
        .rename(columns={"entity_name_before": "entity_name"})
    )

    accepted_norm = set(accepted_df["entity_name"].map(_norm_name))

    # Single output file: all names from pre-BERT list with yes/no acceptance flag.
    out_df = before_df[["entity_name", "entity_type", "entity_name_norm"]].copy()
    out_df["accepted_in_second_search"] = out_df["entity_name_norm"].map(
        lambda n: "yes" if n in accepted_norm else "no"
    )

    accepted_lookup = accepted_df.copy()
    accepted_lookup["entity_name_norm"] = accepted_lookup["entity_name"].map(_norm_name)
    accepted_lookup = accepted_lookup[["entity_name_norm", "suggested_name", "observed_statuses"]].drop_duplicates(
        subset=["entity_name_norm"],
        keep="first",
    )

    out_df = out_df.merge(accepted_lookup, on="entity_name_norm", how="left")
    out_df = out_df[[
        "entity_name",
        "entity_type",
        "accepted_in_second_search",
        "suggested_name",
        "observed_statuses",
    ]]

    out_path = out_dir / f"20260508-{disease_name}_second_search_review.csv"
    out_df.to_csv(out_path, index=False)

    summary = {
        "disease": disease_name,
        "before_count": int(len(before_df)),
        "accepted_yes_count": int((out_df["accepted_in_second_search"] == "yes").sum()),
        "accepted_no_count": int((out_df["accepted_in_second_search"] == "no").sum()),
        "output_path": str(out_path),
    }
    return summary


all_summaries = []
for disease, pair in DISEASE_FILE_PAIRS.items():
    summary = build_outputs_for_disease(
        disease_name=disease,
        before_csv=pair["before"],
        final_csv=pair["final"],
        out_dir=QC_OUTPUTS_DIR,
    )
    all_summaries.append(summary)

summary_df = pd.DataFrame(all_summaries)
summary_df